# Electron Monte Carlo Transport

This notebook controls the settings, data loading, simulation loop, summary, and plots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import electron_mc_transport as emc

## 1. User settings

In [ ]:
# ----------------------------
# Material controls
# ----------------------------

# Change these to use a different target.
# The file should contain LXCat-style cross-section blocks.
CROSS_SECTION_FILE = "eHexsec.txt"
MATERIAL_NAME = "helium gas"
TARGET_NAME = "He"
NUMBER_DENSITY_M3 = 2.5e26       # target number density [particles / m^3]

# ----------------------------
# Simulation controls
# ----------------------------

INITIAL_ENERGY_EV = 100.0        # initial electron energy
NSTEPS = 1000                    # number of Monte Carlo updates
RANDOM_SEED = 1                  # set to None for a new random trajectory each run

ENERGY_CUTOFF_EV = 0.01          # electrons below this energy stop updating

# ----------------------------
# Secondary electron controls
# ----------------------------

TRACK_SECONDARIES = True         # keep secondary electrons from ionization
MAX_ELECTRONS = 100              # safety cap so ionization does not explode forever

# ----------------------------
# Print/plot controls
# ----------------------------

PRINT_EVERY = 100
OUTPUT_PLOT = "electron_trajectory.png" 

In [ ]:
if RANDOM_SEED is not None:
    np.random.seed(RANDOM_SEED)

## 2. Load the cross sections and create the material

In [ ]:
material = emc.load_lxcat_material(
    CROSS_SECTION_FILE,
    material_name=MATERIAL_NAME,
    target_name=TARGET_NAME,
    number_density_m3=NUMBER_DENSITY_M3,
)

print(material)

for species in material.species:
    print(species)

    for process in species.processes:
        print("  ", process)

## 3. Initialize the simulation

In [ ]:
electrons = [
    emc.ElectronParticle(
        energy_eV=INITIAL_ENERGY_EV,
        material=material,
        idx=0,
        energy_cutoff_eV=ENERGY_CUTOFF_EV,
    )
]

next_idx = 1

print("Initial electron:")
print(electrons[0])

## 4. Run the simulation loop

In [ ]:
for step in range(NSTEPS):

    # Make a copy so we can append new secondary electrons safely.
    current_electrons = list(electrons)

    for electron in current_electrons:

        if not electron.alive:
            continue

        secondary, event = electron.update(next_idx=next_idx)

        # Keep the secondary electron if ionization created one.
        if (
            TRACK_SECONDARIES
            and secondary is not None
            and len(electrons) < MAX_ELECTRONS
        ):
            electrons.append(secondary)
            next_idx += 1

    if step % PRINT_EVERY == 0:
        alive = sum(electron.alive for electron in electrons)

        print(
            f"step {step:4d} | "
            f"electrons = {len(electrons):3d} | "
            f"alive = {alive:3d} | "
            f"primary energy = {electrons[0].energy_eV:8.3f} eV"
        )

print("\nSimulation complete.")

## 5. Summary

In [ ]:
primary = electrons[0]

print("Final primary electron:")
print(primary)

print("\nMean free path at final primary energy:")
print(f"lambda = {primary.mean_free_path():.6e} m")

print("\nDistance traveled by primary:")
print(f"{primary.distance_traveled:.6e} m")

print("\nReal collisions for primary:")
print(primary.real_collision_count)

print("\nTotal electrons created:")
print(len(electrons) - 1)

# Count events over all electrons.
event_counts = {}

for electron in electrons:
    for row in electron.history:
        event = row[5]
        event_counts[event] = event_counts.get(event, 0) + 1

print("\nEvent counts across all electrons:")
for event in sorted(event_counts):
    print(f"{event:10s} : {event_counts[event]}")

print("\nFirst few primary trajectory points:")
print("   t [s]        x [m]        y [m]        z [m]      E [eV]      event")

for row in primary.history[:15]:
    print(
        f"{row[0]:10.3e}  "
        f"{row[1]:10.3e}  "
        f"{row[2]:10.3e}  "
        f"{row[3]:10.3e}  "
        f"{row[4]:9.3f}   "
        f"{row[5]}"
    )

## 6. Plot the full trajectory

In [ ]:
plt.figure(figsize=(8, 7))

labels_used = set()
secondary_count = 1

for electron in electrons:
    h = np.array(electron.history, dtype=object)

    if len(h) == 0:
        continue

    x = h[:, 1].astype(float)
    y = h[:, 2].astype(float)

    if electron.idx == 0:
        plt.plot(x, y, "-", linewidth=1.5, label="primary trajectory")

    else:
        plt.plot(
            x, y,
            "-",
            linewidth=0.8,
            alpha=0.35,
            label=f"secondary trajectory {secondary_count}"
        )

        secondary_count += 1

    # Mark excitation, ionization, and attachment events.
    for row in electron.history:
        event = row[5]

        if event == "excitation":
            label = "excitation" if "excitation" not in labels_used else None

            plt.plot(
                row[1], row[2],
                marker="o",
                linestyle="None",
                markersize=5,
                label=label,
            )

            labels_used.add("excitation")

        elif event == "ionization":
            label = "ionization" if "ionization" not in labels_used else None

            plt.plot(
                row[1], row[2],
                marker="s",
                linestyle="None",
                markersize=5,
                label=label,
            )

            labels_used.add("ionization")

        elif event == "attachment":
            label = "attachment" if "attachment" not in labels_used else None

            plt.plot(
                row[1], row[2],
                marker="x",
                linestyle="None",
                markersize=6,
                label=label,
            )

            labels_used.add("attachment")

plt.title(f"Electron Monte Carlo trajectory in {material.name}")
plt.xlabel("x position [m]")
plt.ylabel("y position [m]")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_PLOT, dpi=200)
plt.show()

print("Saved plot to:", OUTPUT_PLOT)

## 7. Full trajectory history

In [ ]:
print("electron   t [s]        x [m]        y [m]        z [m]      E [eV]      event")

for electron in electrons:
    for row in electron.history:
        print(
            f"{electron.idx:8d}  "
            f"{row[0]:10.3e}  "
            f"{row[1]:10.3e}  "
            f"{row[2]:10.3e}  "
            f"{row[3]:10.3e}  "
            f"{row[4]:9.3f}   "
            f"{row[5]}"
        )

## Optional: gas mixture example

This is only a template. You need separate cross-section files for each species.

In [ ]:
# n2_processes = emc.load_lxcat_table("N2_xsecs.txt", target_name="N2")
# o2_processes = emc.load_lxcat_table("O2_xsecs.txt", target_name="O2")
#
# air = emc.make_gas_mixture("air", [
#     ["N2", 0.78 * NUMBER_DENSITY_M3, n2_processes],
#     ["O2", 0.21 * NUMBER_DENSITY_M3, o2_processes],
# ])
#
# print(air)